# TeenPattiBench — Gemma 3 4B

Runs **Gemma 3 4B Instruct** over the 2,000-item TeenPattiBench eval set, writes a
predictions file, and scores it — all in this notebook.

**Nothing to upload.** The eval set, the system prompt and the scorer are pulled
straight from the public repo.

### Before you run
1. `Runtime → Change runtime type → GPU`. **L4 or A100 preferred.** A T4 works but
   has no bfloat16, and Gemma in float16 can overflow — cell 6 checks for that and
   tells you if it happened.
2. Gemma 3 is a **gated** model. Accept the licence at
   <https://huggingface.co/google/gemma-3-4b-it>, then either add your token as a
   Colab secret named `HF_TOKEN` (🔑 in the left sidebar) or let cell 4 prompt you.
3. `Runtime → Run all`.

Roughly 25–40 min on an L4 for all 2,000 items at batch size 8.

### Reusing this for another model
Change `MODEL_ID` and `MODEL_NAME` in cell 1. Nothing else is Gemma-specific — the
chat-template handling in cell 5 probes what the model accepts rather than assuming.

## 1. Configuration — the only cell you normally edit

In [ ]:
MODEL_ID   = "google/gemma-3-4b-it"
MODEL_NAME = "Gemma 3 4B"

# Which eval file to run.
#   "v1"       - the shipped benchmark (fetched from GitHub)
#   "shuffled" - the action-order control; not in the repo yet, so you will be
#                asked to upload data/teenpattibench_eval_shuffled.jsonl
EVAL_VARIANT = "v1"

BATCH_SIZE     = 8       # drop to 4 if you hit out-of-memory
MAX_NEW_TOKENS = 16      # matches the Qwen / Mistral / Llama runs
DTYPE          = "auto"  # "auto" | "bfloat16" | "float16"
LIMIT          = None    # e.g. 50 for a quick trial; None = all 2,000

REPO   = "dbXD320/teenpatti1"
BRANCH = "restructure-and-phase4"
OUT    = "preds_gemma3_4b_colab.jsonl"

# Greedy decoding, so the run is reproducible rather than lucky. Do not change
# without recording it -- the other four runs in this project were all greedy,
# and a sampled run is not comparable to them.
DO_SAMPLE = False

## 2. Install and check the GPU

In [ ]:
# Gemma 3 needs transformers >= 4.50.
!pip -q install -U "transformers>=4.50" accelerate huggingface_hub

import torch, platform, transformers
print("transformers", transformers.__version__, "| torch", torch.__version__)

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> GPU, then run this cell again."
    )

gpu = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
bf16_ok = torch.cuda.is_bf16_supported()
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu}  compute {cap[0]}.{cap[1]}  {total_gb:.1f} GB  bf16={bf16_ok}")

if DTYPE == "auto":
    TORCH_DTYPE = torch.bfloat16 if bf16_ok else torch.float16
else:
    TORCH_DTYPE = getattr(torch, DTYPE)
print("dtype:", TORCH_DTYPE)

if TORCH_DTYPE == torch.float16:
    print(
        "\nNOTE: running in float16 because this GPU has no bfloat16.\n"
        "Gemma was trained in bfloat16 and can overflow to NaN in float16.\n"
        "Cell 6 checks for that explicitly. If it trips, switch to an L4 or A100."
    )

## 3. Fetch the benchmark, the system prompt and the scorer

In [ ]:
import json, hashlib, os, urllib.request

RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"

def fetch(path, dest):
    urllib.request.urlretrieve(f"{RAW}/{path}", dest)
    return dest

fetch("data/teenpattibench_system_prompt.txt", "system_prompt.txt")
fetch("src/metrics.py", "metrics.py")          # stdlib-only, runs anywhere

if EVAL_VARIANT == "v1":
    EVAL_PATH = fetch("data/teenpattibench_eval.jsonl", "eval.jsonl")
elif EVAL_VARIANT == "shuffled":
    from google.colab import files
    print("Upload data/teenpattibench_eval_shuffled.jsonl")
    up = files.upload()
    EVAL_PATH = list(up)[0]
else:
    raise ValueError(f"unknown EVAL_VARIANT {EVAL_VARIANT!r}")

SYSTEM_PROMPT = open("system_prompt.txt").read()
items = [json.loads(l) for l in open(EVAL_PATH) if l.strip()]
if LIMIT:
    items = items[:LIMIT]

EVAL_SHA = hashlib.sha256(open(EVAL_PATH, "rb").read()).hexdigest()

print(f"{len(items):,} items loaded from {EVAL_PATH}")
print(f"eval sha256 : {EVAL_SHA[:16]}...   (records which variant this was)")
print(f"system prompt: {len(SYSTEM_PROMPT)} chars")
print("\n--- one prompt, last 3 lines ---")
print("\n".join(items[0]["prompt"].split("\n")[-3:]))

## 4. Sign in to Hugging Face (Gemma is gated)

In [ ]:
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("Using the HF_TOKEN Colab secret.")
except Exception:
    pass

if token:
    login(token=token)
else:
    print("No HF_TOKEN secret found. Paste a token with read access:")
    login()

## 5. Load the model, and work out how it wants its prompts

Two things are probed rather than assumed, because they differ across families:

* **which class loads it** — Gemma 3 4B is a vision-language checkpoint, so
  `AutoModelForCausalLM` may refuse it;
* **whether it accepts a `system` role** — several Gemma releases reject one and
  require the system text folded into the first user turn. Which path was taken is
  recorded in the output file, because it changes what the run means.

In [ ]:
import importlib
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"                      # required for batched generation
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def _load(cls_name):
    cls = getattr(importlib.import_module("transformers"), cls_name)
    for kw in ("dtype", "torch_dtype"):        # renamed in transformers 4.56
        try:
            return cls.from_pretrained(MODEL_ID, device_map="auto",
                                       **{kw: TORCH_DTYPE})
        except TypeError:
            continue
    raise RuntimeError("neither dtype= nor torch_dtype= accepted")

model, LOADER = None, None
for name in ("Gemma3ForConditionalGeneration", "AutoModelForImageTextToText",
             "AutoModelForCausalLM"):
    try:
        model = _load(name); LOADER = name
        print(f"loaded with {name}")
        break
    except Exception as e:
        print(f"  {name} -> {type(e).__name__}: {str(e)[:110]}")
if model is None:
    raise SystemExit("could not load the model with any known class")
model.eval()

# -- how does this tokenizer want the system prompt? -----------------------
def _try(build):
    try:
        text = tok.apply_chat_template(build(), tokenize=False,
                                       add_generation_prompt=True)
        return text if isinstance(text, str) and text.strip() else None
    except Exception:
        return None

S, U = SYSTEM_PROMPT, items[0]["prompt"]
STYLES = [
    ("system_role", lambda s, u: [{"role": "system", "content": s},
                                  {"role": "user", "content": u}]),
    ("system_role_parts", lambda s, u: [
        {"role": "system", "content": [{"type": "text", "text": s}]},
        {"role": "user",   "content": [{"type": "text", "text": u}]}]),
    ("prepended", lambda s, u: [{"role": "user", "content": s + "\n\n" + u}]),
    ("prepended_parts", lambda s, u: [
        {"role": "user", "content": [{"type": "text", "text": s + "\n\n" + u}]}]),
]

PROMPT_STYLE, _builder = None, None
for name, b in STYLES:
    if _try(lambda: b(S, U)) is not None:
        PROMPT_STYLE, _builder = name, b
        break
if PROMPT_STYLE is None:
    raise SystemExit("no chat-template style worked for this tokenizer")

def build_prompt(user_text):
    return tok.apply_chat_template(_builder(SYSTEM_PROMPT, user_text),
                                   tokenize=False, add_generation_prompt=True)

print(f"\nprompt style: {PROMPT_STYLE}")
if PROMPT_STYLE.startswith("prepended"):
    print("  NOTE: this model rejects a system role, so the glossary is folded into\n"
          "  the user turn. Comparable in content to the other runs, but not an\n"
          "  identical delivery -- it is recorded in the output metadata.")
print("\n--- rendered prompt, first 200 chars ---")
print(build_prompt(U)[:200])

## 6. Smoke test — 5 items before committing to 2,000

Catches the two failures that otherwise waste a full run: float16 overflow
producing empty or garbage output, and a chat template that renders but yields
nothing usable.

In [ ]:
import metrics   # the repo's own parser, so the notebook and the scorer agree

@torch.no_grad()
def generate(batch_texts):
    enc = tok([build_prompt(t) for t in batch_texts], return_tensors="pt",
              padding=True, truncation=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                         do_sample=DO_SAMPLE, pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return tok.batch_decode(gen, skip_special_tokens=True)

probe = items[:5]
raws = generate([i["prompt"] for i in probe])
ok = 0
for it, raw in zip(probe, raws):
    parsed = metrics.parse_action(raw, it["legal_actions"])
    ok += parsed is not None
    print(f"  {it['id']}  raw={raw.strip()[:40]!r:44s} -> {parsed}   "
          f"(key: {it['correct_action']})")

if ok == 0:
    raise SystemExit(
        "None of the 5 probes produced a legal action.\n"
        "If the raw outputs are empty or nonsense and you are on a T4, this is "
        "almost certainly float16 overflow: switch to an L4/A100, or set "
        "DTYPE='float32' (slow, and 4B may not fit)."
    )
print(f"\n{ok}/5 parsed to a legal action -- proceeding.")

## 7. Run all 2,000 items

In [ ]:
import time
from tqdm.auto import tqdm

records, t0 = [], time.time()
for s in tqdm(range(0, len(items), BATCH_SIZE), desc="generating"):
    batch = items[s:s + BATCH_SIZE]
    try:
        raws = generate([i["prompt"] for i in batch])
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise SystemExit("Out of memory. Lower BATCH_SIZE and re-run this cell.")
    for it, raw in zip(batch, raws):
        parsed = metrics.parse_action(raw, it["legal_actions"])
        records.append({
            "id": it["id"],
            # `action` is what src/metrics.py reads, so this file needs NO
            # remapping -- unlike the earlier runs, which emitted `answer`.
            "action": parsed if parsed is not None else raw.strip(),
            "raw": raw.strip(),
        })

elapsed = time.time() - t0
print(f"\n{len(records):,} items in {elapsed/60:.1f} min "
      f"({len(records)/elapsed:.1f} items/s)")

## 8. Write the predictions file

In [ ]:
import datetime

meta = {
    "type": "metadata",
    "model": {
        "name": MODEL_NAME, "model_id": MODEL_ID,
        "architecture": [type(model).__name__], "loader_class": LOADER,
        "num_parameters": int(sum(p.numel() for p in model.parameters())),
        "dtype": str(TORCH_DTYPE), "device": str(model.device),
        "quantized": False,
    },
    "tokenizer": {"name_or_path": MODEL_ID,
                  "vocab_size": int(tok.vocab_size),
                  "padding_side": tok.padding_side},
    "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE,
                   "temperature": None, "top_p": None, "top_k": None,
                   "batch_size": BATCH_SIZE, "thinking_enabled": False},
    "dataset": {"name": "TeenPattiBench", "variant": EVAL_VARIANT,
                "evaluation_file": EVAL_PATH, "sha256": EVAL_SHA,
                "num_questions": len(items)},
    "prompt": {"system_prompt_delivery": PROMPT_STYLE,
               "system_prompt": SYSTEM_PROMPT},
    "runtime": {"gpu": gpu, "torch": torch.__version__,
                "transformers": transformers.__version__,
                "elapsed_seconds": round(elapsed, 1),
                "timestamp_utc": datetime.datetime.now(
                    datetime.timezone.utc).isoformat()},
    "output": {"format": "JSONL", "prediction_file": OUT,
               "action_field": "action"},
}

with open(OUT, "w") as fh:
    fh.write(json.dumps(meta) + "\n")
    for r in records:
        fh.write(json.dumps(r) + "\n")

print(f"wrote {OUT}  ({os.path.getsize(OUT)/1e3:.0f} KB, "
      f"1 metadata line + {len(records):,} predictions)")

## 9. Sanity checks before believing any score

A prediction file can be complete, well-formed and still wrong. These are the
checks that caught real problems in earlier runs.

In [ ]:
import collections

by_id = {i["id"]: i for i in items}
ids = [r["id"] for r in records]

print(f"records                  : {len(records):,}")
print(f"unique ids               : {len(set(ids)):,}   "
      f"{'OK' if len(set(ids)) == len(records) else 'DUPLICATES'}")
print(f"covers every eval item   : {set(ids) == set(by_id)}")

illegal = [r for r in records if r["action"] not in by_id[r["id"]]["legal_actions"]]
print(f"answers not legal for their own item: {len(illegal)}")
for r in illegal[:3]:
    print(f"    {r['id']}: said {r['action']!r}, legal "
          f"{by_id[r['id']]['legal_actions']}")

dist = collections.Counter(r["action"] for r in records)
key = collections.Counter(i["correct_action"] for i in items)
print("\naction distribution")
print(f"  {'action':<12}{'model':>8}{'key':>8}")
for a in ("chaal", "raise", "show", "pack", "see", "stay-blind"):
    print(f"  {a:<12}{dist.get(a, 0):>8}{key.get(a, 0):>8}")
other = {k: v for k, v in dist.items()
         if k not in ("chaal", "raise", "show", "pack", "see", "stay-blind")}
if other:
    print(f"  unparsed / other: {other}")

for a in ("pack", "raise", "show"):
    if dist.get(a, 0) == 0:
        print(f"\n  NOTE: never answered {a!r} in {len(records):,} items. "
              f"That is structural, not a preference -- see cell 11.")

## 10. Score

In [ ]:
preds = {r["id"]: r for r in records}
report = metrics.score(items, preds)
print(metrics.format_report(report, metrics.trivial_baselines(items)))

## 11. The breakdowns that decide what the score means

Three things the headline number hides, all of which changed the reading of
earlier models.

**Per-stratum baselines.** Each section has its own mix of right answers, so each
has its own "answer one word forever" score. Conversion's is 62.96%, not 25.94% —
measuring it against the global baseline turns a 25-point failure into an apparent
success.

**Menu position.** Every v1 prompt lists the actions in the same order, so position
and identity are confounded. Llama-3.2-3B never selected past index 1, which alone
explains its zero raises and zero shows.

**Card blindness.** Whether the answer changes at all when the same betting
situation is dealt a different hand.

In [ ]:
ALL = ("chaal", "pack", "raise", "show", "see", "stay-blind")
graded = [i for i in items if not i["is_mixed"]]

def best_const(sub):
    b = (None, -1)
    for a in ALL:
        h = sum(1 for i in sub if i["correct_action"] == a)
        if h > b[1]:
            b = (a, h)
    return b[0], 100 * b[1] / len(sub)

def aa(sub):
    return 100 * sum(1 for i in sub
                     if preds[i["id"]]["action"] == i["correct_action"]) / len(sub)

print("PER STRATUM, against its OWN baseline")
rows = [("All graded", graded)] + [
    (lab, [i for i in graded if i["meta"]["family"] == f])
    for f, lab in (("seen_bet", "Seen betting"), ("look", "Conversion"),
                   ("blind_bet", "Blind betting"))]
for lab, sub in rows:
    if not sub:
        continue
    bc, bp = best_const(sub)
    print(f"  {lab:<15} n={len(sub):>5}  model {aa(sub):6.2f}%  "
          f"baseline {bc} {bp:6.2f}%   {aa(sub)-bp:+6.1f}")

look = [i for i in graded if i["meta"]["family"] == "look"]
if look:
    print("\nCONVERSION BY TURN  (the benchmark's primary target)")
    print("  A model that always says the same word scores exactly the key's rate.")
    for t in sorted({i["meta"]["turn_index"] for i in look}):
        sub = [i for i in look if i["meta"]["turn_index"] == t]
        k = 100 * sum(1 for i in sub if i["correct_action"] == "see") / len(sub)
        m = 100 * sum(1 for i in sub if preds[i["id"]]["action"] == "see") / len(sub)
        print(f"  turn {t}  n={len(sub):>3}  key says see {k:6.2f}%  "
              f"model says see {m:6.1f}%  AA {aa(sub):6.2f}%")

print("\nMENU POSITION")
bet = [i for i in items if i["legal_actions"][0] != "see"]
lk = [i for i in items if i["legal_actions"][0] == "see"]
pos = collections.Counter(
    i["legal_actions"].index(preds[i["id"]]["action"])
    for i in bet if preds[i["id"]]["action"] in i["legal_actions"])
tot = sum(pos.values()) or 1
print(f"  bet nodes, positions used: {sorted(pos)}  " +
      " ".join(f"pos{k}={100*v/tot:5.1f}%" for k, v in sorted(pos.items())))
if lk:
    p0 = 100 * sum(1 for i in lk
                   if preds[i["id"]]["action"] == i["legal_actions"][0]) / len(lk)
    print(f"  look nodes, picks the first listed option: {p0:5.1f}%")
print("  If the model never selects a high index, its 'never raises / never shows'\n"
      "  is about layout, not the game. Re-run on the shuffled eval to separate them.")

print("\nCARD BLINDNESS  (seen items, nodes offering >= 6 different hands)")
seen = [i for i in graded if i["meta"]["family"] == "seen_bet"]
bn = collections.defaultdict(list)
for i in seen:
    bn[i["meta"]["node"]].append(preds[i["id"]]["action"])
big = {k: v for k, v in bn.items() if len(v) >= 6}
if big:
    const = sum(1 for v in big.values() if len(set(v)) == 1)
    sv = collections.defaultdict(list)
    for i in seen:
        sv[i["meta"]["node"]].append(i["correct_action"])
    sconst = sum(1 for k, v in sv.items() if len(v) >= 6 and len(set(v)) == 1)
    print(f"  model gives one identical answer regardless of its cards: "
          f"{const}/{len(big)}")
    print(f"  the solver does so:                                        "
          f"{sconst}/{len(big)}")

## 12. Download

In [ ]:
from google.colab import files
files.download(OUT)

# To keep it in Drive instead:
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copy(OUT, "/content/drive/MyDrive/" + OUT)

print(f"\nCommit it to the repo as results/{OUT}")
print("It already uses the `action` field, so it scores with no remapping:")
print(f"  python src/metrics.py --eval data/teenpattibench_eval.jsonl \\")
print(f"                        --predictions results/{OUT} \\")
print(f"                        --json runs/report_gemma3_4b.json")